In [11]:
import numpy as np
import tensorflow as tf
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Flatten

In [12]:
import os
import cv2
import matplotlib.pyplot as plt

In [13]:
IMG_SIZE = 128


In [14]:
def load_data(path):
    data=[]
    labels=[]
    categories=['cat','dog']
    for category in categories:
        folder=os.path.join(path,category)
        label=categories.index(category)
        for img in os.listdir(folder):
            img_path=os.path.join(folder,img)
            image=cv2.imread(img_path)
            if image is None:
                continue
            image=cv2.resize(image,(IMG_SIZE,IMG_SIZE))
            data.append(image)   
            labels.append(label)
    return np.array(data),np.array(labels)
         

In [15]:
train_path='train'
val_path='val'

In [16]:
X_train,y_train=load_data(train_path)
X_val,y_val=load_data(val_path)

In [17]:
X_train=X_train/255
X_val=X_val/255


In [18]:
Datagen=ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)
Train_data=Datagen.flow(
    X_train, y_train, batch_size=32, shuffle=True
)
val_datagen=ImageDataGenerator()
Val_data=val_datagen.flow(
    X_val, y_val, batch_size=32, shuffle=False
)


In [19]:
model=Sequential([
    Flatten(input_shape=(128,128,3)),
    Dense(512, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.summary()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_1 (Flatten)         (None, 49152)             0         
                                                                 
 dense_5 (Dense)             (None, 512)               25166336  
                                                                 
 batch_normalization_1 (Bat  (None, 512)               2048      
 chNormalization)                                                
                                                                 
 dropout_2 (Dropout)         (None, 512)               0         
                                                                 
 dense_6 (Dense)             (None, 256)               131328    
                                                                 
 dropout_3 (Dropout)         (None, 256)               0         
                                                      

In [20]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

model.fit(Train_data,epochs=20,batch_size=32,verbose=1,validation_data=Val_data)
test_loss,test_accuracy = model.evaluate(X_val,y_val)
print(test_accuracy,test_loss)


Epoch 1/20
9/9 [==============================] - 4s 261ms/step - loss: 0.6678 - accuracy: 0.6073 - val_loss: 4.7740 - val_accuracy: 0.4143
Epoch 2/20
9/9 [==============================] - 2s 246ms/step - loss: 0.6812 - accuracy: 0.6000 - val_loss: 1.6630 - val_accuracy: 0.6571
Epoch 3/20
9/9 [==============================] - 2s 242ms/step - loss: 0.6489 - accuracy: 0.6618 - val_loss: 1.6876 - val_accuracy: 0.4571
Epoch 4/20
9/9 [==============================] - 2s 203ms/step - loss: 0.6552 - accuracy: 0.6545 - val_loss: 0.8388 - val_accuracy: 0.6571
Epoch 5/20
9/9 [==============================] - 2s 206ms/step - loss: 0.6458 - accuracy: 0.6545 - val_loss: 1.2510 - val_accuracy: 0.4143
Epoch 6/20
9/9 [==============================] - 2s 181ms/step - loss: 0.6761 - accuracy: 0.6582 - val_loss: 0.7548 - val_accuracy: 0.6143
Epoch 7/20
9/9 [==============================] - 2s 214ms/step - loss: 0.6448 - accuracy: 0.6764 - val_loss: 0.7015 - val_accuracy: 0.6571
Epoch 8/20
9/9 [====

In [21]:
from sklearn.metrics import confusion_matrix, classification_report
y_pred = (model.predict(X_val) > 0.5).astype(int)
print(confusion_matrix(y_val, y_pred))
print(classification_report(y_val, y_pred))

3/3 [==============================] - 1s 10ms/step
[[ 2 22]
 [ 2 44]]
              precision    recall  f1-score   support

           0       0.50      0.08      0.14        24
           1       0.67      0.96      0.79        46

    accuracy                           0.66        70
   macro avg       0.58      0.52      0.46        70
weighted avg       0.61      0.66      0.57        70

